In [1]:
import os

from google.cloud import bigquery
client = bigquery.Client(project="expanded-flame-491820-j2")

In [2]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 30.06,     
    "longitude": 31.25,
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "Africa/Cairo"
}

response = requests.get(url, params=params)
data = response.json()

print(data.keys())
print(data["hourly"].keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m'])


In [3]:
import pandas as pd

hourly = data["hourly"]

df = pd.DataFrame({
    "timestamp": hourly["time"],
    "temperature_c": hourly["temperature_2m"],
    "humidity_pct": hourly["relative_humidity_2m"],
    "wind_speed_kmh": hourly["wind_speed_10m"]
})


df["timestamp"] = pd.to_datetime(df["timestamp"])

print(f"Rows: {len(df)}")
print(df.head())

Rows: 168
            timestamp  temperature_c  humidity_pct  wind_speed_kmh
0 2026-04-12 00:00:00           17.7            69             3.1
1 2026-04-12 01:00:00           17.3            71             2.2
2 2026-04-12 02:00:00           16.4            75             2.3
3 2026-04-12 03:00:00           15.5            79             3.1
4 2026-04-12 04:00:00           14.6            81             2.8


In [4]:

df["city"] = "Cairo"
df["country"] = "Egypt"
df["latitude"] = 30.06
df["longitude"] = 31.25


df["comfort_level"] = df["temperature_c"].apply(
    lambda x: "Cold" if x < 15 else "Comfortable" if x < 28 else "Hot"
)


df["hour"] = df["timestamp"].dt.hour
df["time_of_day"] = df["hour"].apply(
    lambda x: "Night" if x < 6 else "Morning" if x < 12 else "Afternoon" if x < 18 else "Evening"
)


df["ingested_at"] = pd.Timestamp.now()

print(df[["timestamp", "temperature_c", "comfort_level", "time_of_day"]].head(10))

            timestamp  temperature_c comfort_level time_of_day
0 2026-04-12 00:00:00           17.7   Comfortable       Night
1 2026-04-12 01:00:00           17.3   Comfortable       Night
2 2026-04-12 02:00:00           16.4   Comfortable       Night
3 2026-04-12 03:00:00           15.5   Comfortable       Night
4 2026-04-12 04:00:00           14.6          Cold       Night
5 2026-04-12 05:00:00           13.9          Cold       Night
6 2026-04-12 06:00:00           13.4          Cold     Morning
7 2026-04-12 07:00:00           14.2          Cold     Morning
8 2026-04-12 08:00:00           17.4   Comfortable     Morning
9 2026-04-12 09:00:00           20.3   Comfortable     Morning


In [5]:
from google.cloud import bigquery

client = bigquery.Client(project="expanded-flame-491820-j2")
PROJECT = "expanded-flame-491820-j2"

table_id = f"{PROJECT}.automobile.cairo_weather"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  
    autodetect=True
)

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()

print(f"Loaded {len(df)} rows into {table_id}")

Loaded 168 rows into expanded-flame-491820-j2.automobile.cairo_weather


In [6]:
import requests
import pandas as pd
from google.cloud import bigquery

def extract():
    
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 30.06,
        "longitude": 31.25,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone": "Africa/Cairo"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()  
    return response.json()

def transform(data):
    
    hourly = data["hourly"]
    
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(hourly["time"]),
        "temperature_c": hourly["temperature_2m"],
        "humidity_pct": hourly["relative_humidity_2m"],
        "wind_speed_kmh": hourly["wind_speed_10m"]
    })
    
    df["city"] = "Cairo"
    df["country"] = "Egypt"
    df["comfort_level"] = df["temperature_c"].apply(
        lambda x: "Cold" if x < 15 else "Comfortable" if x < 28 else "Hot"
    )
    df["time_of_day"] = df["timestamp"].dt.hour.apply(
        lambda x: "Night" if x < 6 else "Morning" if x < 12 else "Afternoon" if x < 18 else "Evening"
    )
    df["ingested_at"] = pd.Timestamp.now()
    
    return df

def load(df):
    
    client = bigquery.Client(project="expanded-flame-491820-j2")
    table_id = "expanded-flame-491820-j2.automobile.cairo_weather"
    
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        autodetect=True
    )
    
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"Loaded {len(df)} rows into {table_id}")

def run_pipeline():
    
    print("Starting pipeline...")
    
    print("Extracting...")
    raw = extract()
    
    print("Transforming...")
    df = transform(raw)
    
    print("Loading...")
    load(df)
    
    print("Pipeline complete!")


run_pipeline()

Starting pipeline...
Extracting...
Transforming...
Loading...
Loaded 168 rows into expanded-flame-491820-j2.automobile.cairo_weather
Pipeline complete!
